In [5]:
import numpy as np
import pandas as pd

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.width', 150)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

# Phương pháp tiếp tuyến (Newton-Raphson)

Điều kiện:

* $(a,b)$ là khoảng cách ly nghiệm

* $f'(x)$ xác định dấu không đổi trên $[a,b]$

* $f''(x)$ xác định dấu không đổi trên $[a,b]$

Điều kiện hội tụ:

* Chọn đúng điểm xuất phát ban đầu $x_0$ (điểm Fourier - $f(x_0) \cdot f''(x_0) > 0$)

# Áp dụng

## 1. Hậu nghiệm

Kiểm tra sai số theo 1 trong 2 công thức sai số sau:
+ $|x_n - x^*| \leqslant \dfrac{|f(x_n)|}{m_1}$ $\quad\quad$ (1)

+ $|x_n - x^*| \leqslant \dfrac{M_2}{2 m_1} \cdot |x_n - x_{n-1}|^2$ $\quad\quad$ (2)

với $m_1 = \min\limits_{[a,b]} |f'(x)|, \quad M_2 = \max\limits_{[a,b]} |f''(x)|$


### 1.1 Thuật toán
**Input:** Hàm số $f(x)$ và các đạo hàm $f'(x), f''(x)$, khoảng cách ly nghiệm $(a,b)$, sai số $\varepsilon$.

**Output:** Nghiệm xấp xỉ $x^*$.

**Thuật toán:**

1. **Khởi tạo:**
   - Nếu $f(a)f(b) > 0 \Rightarrow$ Dừng (khoảng không hợp lệ).
   - Nếu $f(a)=0$ (hoặc $f(b)=0$) $\Rightarrow x^*=a$ (hoặc $b$), dừng.
   - Tính $\displaystyle m_1 = \min_{[a,b]} |f'(x)|$ và $\displaystyle M_2 = \max_{[a,b]} |f''(x)|$.
   - Chọn điểm xuất phát $x_0$: Nếu $f(a)f''(a) > 0 \Rightarrow x_0 = a$, ngược lại $\Rightarrow x_0 = b$.
   - Đặt vòng lặp $i=0$.
2. **Quá trình lặp:**
   - Tính $\displaystyle x_{i+1} = x_i - \frac{f(x_i)}{f'(x_i)}$.
   - Tính sai số $\Delta_i$:
     + Theo công thức 1: $\displaystyle \Delta_i = \frac{|f(x_{i+1})|}{m_1}$.
     + Theo công thức 2: $\displaystyle \Delta_i = \frac{M_2}{2m_1}|x_{i+1} - x_i|^2$.
   - Nếu $f(x_{i+1}) = 0$ hoặc $\Delta_i < \varepsilon \Rightarrow x^* = x_{i+1}$, kết thúc thuật toán.
   - Gán $x_i \leftarrow x_{i+1}$, tăng $i \leftarrow i+1$.

### 1.2 Code
#### 1.2.1 Có cộng thêm sai số làm tròn
##### Công thức 1

In [ ]:
def newton_iteration_v1 (f, df, d2f, a, b, n, rbl):	
	M2 = max([np.abs(d2f(x)) for x in [a, b]]) #M2 is the maximum value of |f''(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print (f"m1 = {m1}, M2 = {M2}")

	#Starting values
	print(f"f(a) = {f(a)}, f''(a) = {d2f(a)}")
	if f(a) * d2f(a) > 0:
		x = a
		print(f"f(a) * f''(a) > 0 => Chọn x_0 = a = {a}")
	elif f(b) * d2f(b) > 0:
		x = b
		print(f"f(a) * f''(a) < 0 => Chọn x_0 = b = {b}")
	
	results = []
	results.append({
        'n': 0,
        'x_n': x,
        'f(x_n)': f(x),
        'delta_x=|f(x_n)| / m_1': abs(f(x)) / m1
    })

	#Newton's method
	x_new = 0; delta_x = 0
	for i in range(n):
		# calculate the next value of x
		x_new = x - f(x) / df(x)
		delta_x = abs(f(x_new)) / m1

		results.append({
            'n': i+1,
            'x_n': x_new,
            'f(x_n)': f(x_new),
            'delta_x=|f(x_n)| / m_1': delta_x
        })

		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break

	#Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))

	if rbl == None:
		print(f"The value of root is: {x}")
	else:
		total_delta = delta_x + 0.5 * 10**(-rbl)
		print(f"The value of root with {rbl} decimal point is: {round(x, rbl)}")
		print(f"Relative error is: {total_delta}")

In [ ]:
f = lambda x: x**4-27
df = lambda x: 4*x**3
d2f = lambda x: 12*x**2

a = 2
b = 3

n = 5
rbl = 9

newton_iteration_v1 (f, df, d2f, a, b, n, rbl)

#### Công thức 2

In [ ]:
def newton_iteration_v2 (f, df, d2f, a, b, n, rbl):	
	M2 = max([np.abs(d2f(x)) for x in [a, b]]) #M2 is the maximum value of |f''(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print (f"m1 = {m1}, M2 = {M2}")

	#Starting values
	print(f"f(a) = {f(a)}, f''(a) = {d2f(a)}")
	if f(a) * d2f(a) > 0:
		x = a
		print(f"f(a) * f''(a) > 0 => Chọn x_0 = a = {a}")
	elif f(b) * d2f(b) > 0:
		x = b
		print(f"f(a) * f''(a) < 0 => Chọn x_0 = b = {b}")
	
	results = []
	results.append({
        'n': 0,
        'x_n': x,
        'f(x_n)': f(x),
        'delta_x=M_2/(2*m_1) * |x_n - x_(n-1)|^2': None
    })

	#Newton's method
	x_new = 0; delta_x = 0;
	for i in range(n):
		# calculate the next value of x
		x_new = x - f(x) / df(x)
		delta_x = M2 / (2 * m1) * (x_new - x)**2

		results.append({
            'n': i+1,
            'x_n': x_new,
            'f(x_n)': f(x_new),
            'delta_x=M_2/(2*m_1) * |x_n - x_(n-1)|^2': delta_x
        })

		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break

	#Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))

	if rbl == None:
		print(f"The value of root is: {x}")
	else:
		total_delta = delta_x + 0.5 * 10**(-rbl)
		print(f"The value of root with {rbl} decimal point is: {round(x, rbl)}")
		print(f"Relative error is: {total_delta}")

In [ ]:
f = lambda x: x**4-27
df = lambda x: 4*x**3
d2f = lambda x: 12*x**2

a = 2
b = 3

n = 5
rbl = 9

newton_iteration_v2 (f, df, d2f, a, b, n, rbl)

#### 1.2.2 Không cộng thêm sai số làm tròn

Thiết lập điều kiện dừng theo 1 trong 2 công thức sai số sau:
+ $|x_n - x^*| \leqslant \dfrac{|f(x_n)|}{m_1} < \epsilon$ $\quad\quad (1)$

+ $|x_n - x^*| \leqslant \dfrac{M_2}{2 m_1} \cdot |x_n - x_{n-1}|^2 < \epsilon$ $\quad\quad (2)$

với $m_1 = \min\limits_{[a,b]} |f'(x)|, \quad M_2 = \max\limits_{[a,b]} |f''(x)|$


#### Công thức 1

Từ công thức (1) suy ra Điều kiện dừng: $|f(x_n)| < m_1 \varepsilon$

In [ ]:
def newton_recursion_absolute_v1 (f, df, d2f, a, b, eps):	
	M2 = max([np.abs(d2f(x)) for x in [a, b]]) #M2 is the maximum value of |f''(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print (f"m1 = {m1}, M2 = {M2}")

	#Starting values
	print(f"f(a) = {f(a)}, f''(a) = {d2f(a)}")
	if f(a) * d2f(a) > 0:
		x = a
		print(f"f(a) * f''(a) > 0 => Chọn x_0 = a = {a}")
	elif f(b) * d2f(b) > 0:
		x = b
		print(f"f(a) * f''(a) < 0 => Chọn x_0 = b = {b}")
	
	results = []
	results.append({
        'n': 0,
        'x_n': x,
        'f(x_n)': f(x),
        'delta_x=|f(x_n)|': abs(f(x))
    })

	#Newton's method
	new_eps = m1*eps
	print(f"delta_x = {new_eps}")

	i=0
	while True:
		# calculate the next value of x
		x_new = x - f(x) / df(x)
		current_delta_x = abs(f(x_new))

		results.append({
            'n': i+1,
            'x_n': x_new,
            'f(x_n)': f(x_new),
            'delta_x=|f(x_n)|': current_delta_x
        })
		
		# update the value of interval 
		x = x_new
		if (f(x_new) == 0): 
			break

		#stop condition
		if current_delta_x < new_eps:
			break
		else:
			i += 1

	#Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))


	print(f"The value of root with absolute error {eps} is: {x}")

Bài tập VD trên lớp:

In [ ]:
f = lambda x: np.exp(x)-4*np.cos(x/2)+1
df = lambda x: np.exp(x)+2*np.sin(x/2)
d2f = lambda x: np.exp(x)+np.cos(x/2)

a = -3
b = -2

eps = 0.5 * pow(10, -7)

newton_recursion_absolute_v1 (f, df, d2f, a, b, eps)

Câu 8.1. Viết thuật toán tính gần đúng hằng số $e$ với sai số $\varepsilon$ cho trước. Áp dụng tính $e$ với bảy chữ số đáng tin sau dấu phẩy.

Câu 8.2. Viết thuật toán tính gần đúng hằng số $\pi$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\pi$ với bảy chữ số đáng tin sau dấu phẩy.

Câu 9. Viết thuật toán tính gần đúng $\sqrt[n]{a}, a \in \mathbb{R}, n \in \mathbb{N}$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\sqrt[5]{17}$ với 6 chữ số đáng tin.

Câu 10. Tìm nghiệm âm lớn nhất của phương trình $e^x-\cos 2x=0$ với 8 chữ số đáng tin bằng phương pháp tiếp tuyến.

#### Công thức 2

Từ công thức (2) suy ra Điều kiện dừng: $|x_n - x_{n-1}| < \sqrt{2 \varepsilon \cdot \dfrac{m_1}{M_2}}$

In [16]:
def newton_recursion_absolute_v2 (f, df, d2f, a, b, eps):	
	M2 = max([np.abs(d2f(x)) for x in [a, b]]) #M2 is the maximum value of |f''(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print (f"m1 = {m1}, M2 = {M2}")

	#Starting values
	print(f"f(a) = {f(a)}, f''(a) = {d2f(a)}")
	if f(a) * d2f(a) > 0:
		x = a
		print(f"f(a) * f''(a) > 0 => Chọn x_0 = a = {a}")
	elif f(b) * d2f(b) > 0:
		x = b
		print(f"f(a) * f''(a) < 0 => Chọn x_0 = b = {b}")
	
	results = []
	results.append({
        'i': 0,
        'x_i': x,
        'f(x_i)': f(x),
        'delta_x=|x_i-x_(i-1)|': None
    })

	#Newton's method
	new_eps = np.sqrt(eps * 2 * m1 / M2)
	print(f"delta_x = {new_eps}")

	i=0
	while True:
		# calculate the next value of x
		x_new = x - f(x) / df(x)
		current_delta_x = abs(x-x_new)

		results.append({
            'i': i+1,
            'x_i': x_new,
            'f(x_i)': f(x_new),
            'delta_x=|x_i-x_(i-1)|': current_delta_x
        })
		
		# update the value of interval 
		x = x_new
		if (f(x_new) == 0): 
			break

		#stop condition
		if current_delta_x < new_eps:
			break
		else:
			i += 1

	#Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))


	print(f"The value of root with absolute error {eps} is: {x}")

Bài tập VD trên lớp:

In [ ]:
f = lambda x: np.exp(x)-4*np.cos(x/2)+1
df = lambda x: np.exp(x)+2*np.sin(x/2)
d2f = lambda x: np.exp(x)+np.cos(x/2)

a = -3
b = -2

eps = 0.5 * pow(10, -7)

newton_recursion_absolute_v2 (f, df, d2f, a, b, eps)

Câu 8.1. Viết thuật toán tính gần đúng hằng số $e$ với sai số $\varepsilon$ cho trước. Áp dụng tính $e$ với bảy chữ số đáng tin sau dấu phẩy.

Câu 8.2. Viết thuật toán tính gần đúng hằng số $\pi$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\pi$ với bảy chữ số đáng tin sau dấu phẩy.

Câu 9. Viết thuật toán tính gần đúng $\sqrt[n]{a}, a \in \mathbb{R}, n \in \mathbb{N}$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\sqrt[5]{17}$ với 6 chữ số đáng tin.

Câu 10. Tìm nghiệm âm lớn nhất của phương trình $e^x-\cos 2x=0$ với 8 chữ số đáng tin bằng phương pháp tiếp tuyến.

### 1.2.3 Tìm nghiệm thoả mãn điều kiện sai số tương đối


Thiết lập điều kiện dừng theo công thức sai số: $\dfrac{|x_n - x^*|}{|x_n|} \leqslant \dfrac{M_2}{2 m_1} \cdot \dfrac{|x_n - x_{n-1}|^2}{|x_n|} < \eta$

với $m_1 = \min\limits_{[a,b]} |f'(x)|, \quad M_2 = \max\limits_{[a,b]} |f''(x)|$

Từ công thức sai số hậu nghiệm, ta có Điều kiện dừng: $\dfrac{|x_n - x_{n-1}|^2}{|x_n|} < 2 \eta \cdot \dfrac{m_1}{M_2}$

In [12]:
def newton_recursion_relative (f, df, d2f, a, b, eta):	
	M2 = max([np.abs(d2f(x)) for x in [a, b]]) #M2 is the maximum value of |f''(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print (f"m1 = {m1}, M2 = {M2}")

	#Starting values
	print(f"f(a) = {f(a)}, f''(a) = {d2f(a)}")
	if f(a) * d2f(a) > 0:
		x = a
		print(f"f(a) * f''(a) > 0 => Chọn x_0 = a = {a}")
	elif f(b) * d2f(b) > 0:
		x = b
		print(f"f(a) * f''(a) < 0 => Chọn x_0 = b = {b}")
	
	results = []
	results.append({
        'i': 0,
        'x_i': x,
        'f(x_i)': f(x),
        'sigma_x=|x_i-x_(i-1)|^2/|x_i|': None
    })

	#Newton's method
	new_eta = 2 * eta * m1 / M2
	print(f"sigma_x = {new_eta}")

	i=0
	while True:
		# calculate the next value of x
		x_new = x - f(x) / df(x)
		current_sigma_x = (x-x_new)**2 / abs(x_new)

		results.append({
            'i': i + 1,
            'x_i': x_new,
            'f(x_i)': f(x_new),
            'sigma_x=|x_i-x_(i-1)|^2/|x_i|': current_sigma_x
        })
		
		# update the value of interval 
		x = x_new
		if (f(x_new) == 0): 
			break

		#stop condition
		if current_sigma_x < new_eta:
			break
		else:
			i += 1

	#Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))


	print(f"The value of root with relative error {eta} is: {x}")

Câu 11. Giải phương trình với nghiệm có sai lệch không vượt quá 0.05% bằng phương pháp tiếp tuyến.

a. $x^5-3x^3+2x^2-x+5=0$

In [ ]:
f = lambda x: x**5 - 3*x**3 + 2*x**2 - x + 5
df = lambda x: 5*x**4 - 9*x**2 + 4*x - 1
d2f = lambda x: 20*x**3 - 18*x + 4

a = -3
b = -2

eta = 0.05 * pow(10, -2)

newton_recursion_relative (f, df, d2f, a, b, eta)

b. $2^x-5x+\sin x = 0$

Nghiệm $x_1$:

Nghiệm $x_2$:

c. $e^{-x}-x=0$